# Portée des Variables en Python : Locales, Globales, `global` et `nonlocal`

Dans cette section, nous allons explorer la **distinction entre variables locales et globales**, ainsi que l’utilisation des mots-clés **`global`** et **`nonlocal`** pour gérer les états dans les fonctions, notamment les fonctions imbriquées. 

## Qu’est-ce que la Portée ?
La **portée** (ou *scope*) d’une variable détermine où elle est accessible dans le code. Python suit la règle **LEGB** (*Local, Enclosing, Global, Built-in*)

## Variables Locales vs Globales

### Définitions
- **Locale** : Définie dans une fonction, accessible uniquement à l’intérieur de cette fonction.
- **Globale** : Définie en dehors de toute fonction, accessible partout dans le module.

### Règles
- Une variable est locale par défaut si elle est assignée dans une fonction.
- Pour modifier une variable globale dans une fonction, il faut utiliser `global`.

### Exemple Simple
Illustrons la distinction.

In [1]:
# Variable globale
compteur = 0

def incrementer():
    """Tente d’incrémenter une variable."""
    # Variable locale par défaut (pas de référence à la globale sans `global`)
    compteur = 1  # Nouvelle variable locale, pas la globale
    print("Compteur local :", compteur)


In [2]:
print("Avant :", compteur)  

Avant : 0


In [3]:
incrementer()   

Compteur local : 1


In [4]:
print("Après :", compteur)

Après : 0



- **`compteur` global** : Défini à 0 en dehors de la fonction.
- **`compteur` local** : Dans `incrementer()`, l’assignation crée une nouvelle variable locale, laissant la globale intacte.
- **Problème** : Sans `global`, la fonction ne peut pas modifier la variable globale.

Corrigeons cela avec `global` !

## Utilisation de `global`

Le mot-clé **`global`** permet à une fonction de modifier une variable globale.

### Syntaxe
```python
global nom_variable
```

### Exemple

Modifions la variable globale.

In [5]:
# Variable globale
compteur = 0

def incrementer():
    """Incrémente la variable globale compteur."""
    global compteur  # Déclare que nous utilisons la globale
    compteur += 1
    print("Compteur dans la fonction :", compteur)


In [6]:
print("Avant :", compteur)  

Avant : 0


In [7]:
incrementer()

Compteur dans la fonction : 1


In [8]:
print("Après :", compteur)


Après : 1


In [9]:
incrementer()

Compteur dans la fonction : 2


In [10]:
print("Après encore :", compteur)

Après encore : 2



- **`global compteur`** : Indique que la fonction modifie la variable globale.
- **Résultat** : Chaque appel à `incrementer()` met à jour `compteur` globalement.
- **Attention** : L’abus de `global` peut rendre le code difficile à suivre ; préférez les retours de fonctions quand possible.

## Variables dans les Fonctions Imbriquées et `nonlocal`

Dans une fonction imbriquée, une variable peut être :
- **Locale** : Définie dans la fonction interne.
- **Enclosing** : Définie dans la fonction englobante.
- **Globale** : Définie à l’extérieur.

Le mot-clé **`nonlocal`** permet de modifier une variable de la portée englobante (*enclosing scope*).

### Exemple Simple
Créons une fonction avec une closure.

In [11]:
def externe():
    """Fonction externe avec une variable locale."""
    compteur = 0
    
    def interne():
        """Tente de modifier la variable englobante."""
        compteur = 1  # Crée une nouvelle variable locale
        print("Compteur interne :", compteur)
    
    interne()
    print("Compteur externe :", compteur)


In [12]:
externe()

Compteur interne : 1
Compteur externe : 0



- **`compteur` dans `externe`** : Variable locale à la portée englobante (0).
- **`compteur` dans `interne`** : Nouvelle variable locale (1), ne modifie pas celle de `externe`.
- **Problème** : Sans `nonlocal`, `interne` ne peut pas accéder à la variable englobante pour la modifier.

Corrigeons avec `nonlocal` !

In [13]:
def externe():
    """Fonction externe avec une variable modifiable."""
    compteur = 0
    
    def interne():
        """Modifie la variable englobante."""
        nonlocal compteur  # Référence la variable de la portée englobante
        compteur += 1
        print("Compteur interne :", compteur)
    
    interne()
    print("Compteur externe :", compteur)

In [14]:
externe()

Compteur interne : 1
Compteur externe : 1



- **`nonlocal compteur`** : Permet à `interne` de modifier la variable `compteur` de `externe`.
- **Résultat** : La modification est visible dans la portée englobante.
- **Limitation** : `nonlocal` ne fonctionne pas pour les variables globales, uniquement pour les portées englobantes.

Comparons `global` et `nonlocal` dans un exemple avancé !

## Utilisation en Machine Learning et Data Science

On retrouve rarement du code utilisant les notions `global` et `nonlocal` en data science, mais cela peut intervenir en créant du code "multi-threading" ou bien dans des systemes gerant des signaux et des interuptions

Ces outils sont puissants pour gérer l’état, mais utilisez-les avec parcimonie pour éviter des dépendances complexes. Expérimentez pour bien comprendre les subtilités des portées !

### Exemple 1: Simulation d'un signal d'arret

In [15]:
import signal
import time

# Variable globale permettant de savoir si on doit s'arrêter
STOP_REQUESTED = False

def handle_sigint(signum, frame):
    global STOP_REQUESTED
    print("\nSignal reçu ! Demande d'arrêt propre...")
    STOP_REQUESTED = True

# On associe SIGINT (Ctrl+C) au handler
signal.signal(signal.SIGINT, handle_sigint)

def long_data_processing():
    print("Début du traitement… (Ctrl+C pour demander un arrêt propre)")
    for i in range(1, 1000000):
        if STOP_REQUESTED:
            print("Arrêt propre effectué. Sauvegarde de l'état…")
            # ex : sauvegarde du checkpoint
            break
        
        # Simulation d'un traitement
        print(f"Traitement de l'item {i}")
        time.sleep(0.3)

    print("Fin du programme.")

long_data_processing()

Début du traitement… (Ctrl+C pour demander un arrêt propre)
Traitement de l'item 1
Traitement de l'item 2
Traitement de l'item 3
Traitement de l'item 4
Traitement de l'item 5
Traitement de l'item 6
Traitement de l'item 7
Traitement de l'item 8
Traitement de l'item 9
Traitement de l'item 10
Traitement de l'item 11
Traitement de l'item 12
Traitement de l'item 13
Traitement de l'item 14
Traitement de l'item 15
Traitement de l'item 16
Traitement de l'item 17
Traitement de l'item 18
Traitement de l'item 19
Traitement de l'item 20
Traitement de l'item 21
Traitement de l'item 22
Traitement de l'item 23
Traitement de l'item 24
Traitement de l'item 25
Traitement de l'item 26
Traitement de l'item 27
Traitement de l'item 28
Traitement de l'item 29
Traitement de l'item 30
Traitement de l'item 31
Traitement de l'item 32
Traitement de l'item 33
Traitement de l'item 34
Traitement de l'item 35
Traitement de l'item 36
Traitement de l'item 37
Traitement de l'item 38
Traitement de l'item 39
Traitement de

### Exemple 2: Systeme_state

In [16]:
# --- État global du système ---
SYSTEM_STATE = {
    "running": False,
    "current_user": None,
    "events": []
}

# --- Fonction pour démarrer le système ---
def start_system(user):
    global SYSTEM_STATE
    SYSTEM_STATE["running"] = True
    SYSTEM_STATE["current_user"] = user
    SYSTEM_STATE["events"].append(f"System started by {user}")

# --- Fonction pour enregistrer un événement ---
def log_event(event):
    global SYSTEM_STATE
    if SYSTEM_STATE["running"]:
        SYSTEM_STATE["events"].append(event)
    else:
        raise RuntimeError("System is not running.")

# --- Fonction pour arrêter le système ---
def stop_system():
    global SYSTEM_STATE
    SYSTEM_STATE["events"].append("System stopped")
    SYSTEM_STATE["running"] = False

# --- Exemple d'utilisation ---
start_system("alice")
log_event("Loading configuration")
log_event("Initializing modules")
stop_system()

print("=== SYSTEM REPORT ===")
print(SYSTEM_STATE)


=== SYSTEM REPORT ===
{'running': False, 'current_user': 'alice', 'events': ['System started by alice', 'Loading configuration', 'Initializing modules', 'System stopped']}


### Exemple 3 : Structure Producteur / Consommateur

In [ ]:
BUFFER = []          # Global: zone partagée
MAX_BUFFER_SIZE = 5  # Limite logique pour montrer le mécanisme

def produce(item):
    global BUFFER
    if len(BUFFER) < MAX_BUFFER_SIZE:
        BUFFER.append(item)
        print(f"Produit → {item}")
    else:
        print("Buffer plein — production en attente")

def consume():
    global BUFFER
    if BUFFER:
        item = BUFFER.pop(0)
        print(f"Consommé ← {item}")
        return item
    else:
        print("Buffer vide — consommation en attente")

# Démonstration
produce("A")
produce("B")
consume()
produce("C")
consume()
consume()


### Exemple 4: Cache Global

In [17]:
CACHE = {}

def load_resource(name):
    global CACHE
    if name not in CACHE:
        print("Création de", name)
        CACHE[name] = f"<resource {name}>"
    else:
        print("Chargement de", name)
    return CACHE[name]

In [18]:
CACHE

{}

In [19]:
r1 = load_resource("model")

Création de model


In [22]:
CACHE

{'model': '<resource model>'}

In [21]:
r2 = load_resource("model")

Chargement de model


In [25]:
r3 = load_resource("data_loader")

Chargement de data_loader


In [24]:
CACHE

{'model': '<resource model>', 'data_loader': '<resource data_loader>'}

## Conclusion

Cette section vous a permis de maîtriser :
- La **distinction entre variables locales et globales** et leurs portées.
- L’utilisation de **`global`** pour modifier des variables globales dans une fonction.
- L’utilisation de **`nonlocal`** pour gérer les variables des portées englobantes dans des fonctions imbriquées.

